In [1]:
from pathlib import Path
import os
import json
import math
import random
import hashlib
import numpy as np
import pandas as pd

## 1. Reproducibility and project paths

In [2]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

def find_project_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    markers = ["pyproject.toml", ".git"]

    for candidate in [start, *start.parents]:
        if any((candidate / marker).exists() for marker in markers):
            return candidate

    return start

PROJECT_ROOT = find_project_root()
print("PROJECT_ROOT:", PROJECT_ROOT)

DEFAULT_SHARED_ROOT = Path(
    os.environ.get(
        "AUTOVISION_SHARED_ROOT",
        r"G:\.shortcut-targets-by-id\1OOvt005TduvEA8h6_snfuyQtbQAjo-3J"
        r"\capstone-vehicle-metadata",
    )
)

OUTPUT_DIR = PROJECT_ROOT / "reports" / "final_dataset_preparation"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("OUTPUT_DIR:", OUTPUT_DIR)

PROJECT_ROOT: C:\Users\elesh\Academics\UTS\36127InnovationLab\Development\AutoVision_Model_Development
OUTPUT_DIR: C:\Users\elesh\Academics\UTS\36127InnovationLab\Development\AutoVision_Model_Development\reports\final_dataset_preparation


## 2. Source-file configuration and helper functions

In [21]:
WEB_METADATA_CANDIDATES = [
    PROJECT_ROOT / "data" / "processed" / "compcars_modeling_ready_v2.csv",
    PROJECT_ROOT / "data" / "processed" / "compcars_modeling_ready.csv",
    PROJECT_ROOT / "data" / "interim" / "compcars_modeling_ready.csv",
    DEFAULT_SHARED_ROOT / "data" / "compcars_kaggle" / "compcars_modeling_ready.csv",
    DEFAULT_SHARED_ROOT / "data" / "compcars_modeling_ready.csv",
]

COLOUR_METADATA_CANDIDATES = [
    PROJECT_ROOT / "data" / "processed" / "compcars_surveillance_inventory_v2.csv",
    PROJECT_ROOT / "data" / "interim" / "compcars_surveillance_inventory_v2.csv",
    DEFAULT_SHARED_ROOT / "data" / "compcars_sv" / "compcars_surveillance_inventory_v2.csv",
    DEFAULT_SHARED_ROOT / "data" / "compcars_surveillance_inventory_v2.csv",
]

CARDD_ROOT = (
    DEFAULT_SHARED_ROOT
    / "data"
    / "cardd_kaggle"
    / "extracted"
)

CLEAN_NORMAL_ROOT = Path(
    os.environ.get(
        "CLEAN_NORMAL_ROOT",
        str(DEFAULT_SHARED_ROOT / "data" / "comprehensive-car-damage"),
    )
)

def first_existing(candidates):
    for p in candidates:
        p = Path(p)
        if p.exists():
            return p
    return None

def repo_search(filename):
    matches = list(PROJECT_ROOT.rglob(filename))
    return matches[0] if matches else None

WEB_METADATA = first_existing(WEB_METADATA_CANDIDATES) or repo_search("compcars_modeling_ready.csv")
COLOUR_METADATA = first_existing(COLOUR_METADATA_CANDIDATES) or repo_search("compcars_surveillance_inventory_v2.csv")

print("WEB_METADATA:", WEB_METADATA)
print("COLOUR_METADATA:", COLOUR_METADATA)
print("CARDD_ROOT:", CARDD_ROOT)
print("CLEAN_NORMAL_ROOT:", CLEAN_NORMAL_ROOT)


WEB_METADATA: C:\Users\elesh\Academics\UTS\36127InnovationLab\Development\AutoVision_Model_Development\data\processed\compcars_modeling_ready_v2.csv
COLOUR_METADATA: C:\Users\elesh\Academics\UTS\36127InnovationLab\Development\AutoVision_Model_Development\data\processed\compcars_surveillance_inventory_v2.csv
CARDD_ROOT: G:\.shortcut-targets-by-id\1OOvt005TduvEA8h6_snfuyQtbQAjo-3J\capstone-vehicle-metadata\data\cardd_kaggle\extracted
CLEAN_NORMAL_ROOT: G:\.shortcut-targets-by-id\1OOvt005TduvEA8h6_snfuyQtbQAjo-3J\capstone-vehicle-metadata\data\comprehensive-car-damage


In [4]:
def deterministic_train_val_split(
    df,
    group_col,
    train_ratio=0.90,
    val_ratio=0.10,
    seed=42,
):
    """Deterministic per-class train/validation split."""
    assert abs(train_ratio + val_ratio - 1.0) < 1e-9

    rng = np.random.default_rng(seed)
    parts = []

    for group_name, group in df.groupby(group_col, sort=True):
        idx = group.index.to_numpy().copy()
        rng.shuffle(idx)

        n = len(idx)
        if n < 2:
            raise ValueError(
                f"Class {group_name!r} has only {n} sample(s); "
                "cannot create train and validation subsets."
            )

        n_val = max(1, int(round(n * val_ratio)))
        n_train = n - n_val

        if n_train < 1:
            n_train = 1
            n_val = n - 1

        g = group.loc[idx].copy()
        g["split"] = ["train"] * n_train + ["val"] * n_val
        parts.append(g)

    return pd.concat(parts, ignore_index=True)


In [5]:
# Generic helpers
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def normalise_slashes(value):
    return str(value).replace("\\", "/")

def portable_suffix(raw_path, markers):
    s = normalise_slashes(raw_path)

    for marker in markers:
        marker = normalise_slashes(marker).strip("/")
        token = marker + "/"
        idx = s.lower().find(token.lower())
        if idx >= 0:
            return s[idx + len(token):].lstrip("/")

    p = Path(s)
    if not p.is_absolute() and ":" not in s[:3]:
        return s.lstrip("./")

    return Path(s).name

def add_portable_path_column(df, source_col, dataset_key, markers):
    out = df.copy()
    out["source_file_path"] = out[source_col].astype(str)
    out["relative_path"] = out[source_col].map(
        lambda x: portable_suffix(x, markers)
    )
    out["dataset_key"] = dataset_key
    return out

def deterministic_group_split(
    df,
    group_col,
    train_ratio=0.80,
    val_ratio=0.10,
    test_ratio=0.10,
    seed=42,
):
    assert abs(train_ratio + val_ratio + test_ratio - 1.0) < 1e-9

    rng = np.random.default_rng(seed)
    parts = []

    for group_name, group in df.groupby(group_col, sort=True):
        idx = group.index.to_numpy().copy()
        rng.shuffle(idx)

        n = len(idx)

        if n < 3:
            raise ValueError(
                f"Class {group_name!r} has only {n} samples; "
                "cannot place it in train, val and test."
            )

        n_val = max(1, int(round(n * val_ratio)))
        n_test = max(1, int(round(n * test_ratio)))
        n_train = n - n_val - n_test

        if n_train < 1:
            n_train = 1
            remaining = n - n_train
            n_val = max(1, remaining // 2)
            n_test = remaining - n_val

        split = np.array(["train"] * n, dtype=object)
        split[n_train:n_train + n_val] = "val"
        split[n_train + n_val:] = "test"

        g = group.loc[idx].copy()
        g["split"] = split
        parts.append(g)

    return pd.concat(parts, ignore_index=True)

def print_split_summary(df, label_col, title):
    print("\n" + "=" * 80)
    print(title)
    print("=" * 80)
    print("Rows:", len(df))
    print("Classes:", df[label_col].nunique())
    print("\nSplit counts:")
    print(df["split"].value_counts())
    print("\nClass count summary:")
    print(df.groupby(label_col).size().describe())

def save_csv(df, name):
    path = OUTPUT_DIR / name
    df.to_csv(path, index=False)
    print("Saved:", path)
    return path

def exact_duplicate_report(df, hash_col="image_hash"):
    if hash_col not in df.columns:
        return None

    hashes = df[hash_col].dropna().astype(str)
    duplicates = hashes.duplicated(keep=False).sum()
    print(f"Rows involved in duplicate hashes: {duplicates}")
    return duplicates



## 3. Final make/model dataset


In [8]:
MIN_IMAGES_PER_MAKE_MODEL = 50

if WEB_METADATA is None:
    raise FileNotFoundError(
        "Could not find compcars_modeling_ready.csv. "
        "Set WEB_METADATA to its actual path in Section 2."
    )

web_df = pd.read_csv(WEB_METADATA)

print("Columns:", web_df.columns.tolist())
print("Rows loaded:", len(web_df))

required = {"file_path", "make", "model_name"}
missing = required - set(web_df.columns)
if missing:
    raise KeyError(f"Missing required columns in web metadata: {sorted(missing)}")

make_model_df = web_df.dropna(subset=["file_path", "make", "model_name"]).copy()

make_model_df["make"] = make_model_df["make"].astype(str).str.strip()
make_model_df["model_name"] = make_model_df["model_name"].astype(str).str.strip()
make_model_df["make_model"] = (
    make_model_df["make"] + " || " + make_model_df["model_name"]
)

if "image_hash" in make_model_df.columns:
    before = len(make_model_df)
    make_model_df = make_model_df.drop_duplicates(subset=["image_hash"]).copy()
    print("Exact duplicate rows removed:", before - len(make_model_df))

class_counts = make_model_df["make_model"].value_counts()
eligible_classes = class_counts[class_counts >= MIN_IMAGES_PER_MAKE_MODEL].index

make_model_df = make_model_df[
    make_model_df["make_model"].isin(eligible_classes)
].copy()

print("Retained make/model classes:", make_model_df["make_model"].nunique())
print("Retained images:", len(make_model_df))
print("Minimum retained class size:", make_model_df["make_model"].value_counts().min())
print("Maximum retained class size:", make_model_df["make_model"].value_counts().max())

make_model_df = add_portable_path_column(
    make_model_df,
    source_col="file_path",
    dataset_key="compcars_web",
    markers=[
        "data/compcars_kaggle/extracted",
        "compcars_kaggle/extracted",
        "data/compcars_web",
        "compcars_web",
    ],
)

make_model_final = deterministic_group_split(
    make_model_df,
    group_col="make_model",
    train_ratio=0.80,
    val_ratio=0.10,
    test_ratio=0.10,
    seed=SEED,
)

make_model_classes = sorted(make_model_final["make_model"].unique())
make_model_to_id = {name: i for i, name in enumerate(make_model_classes)}
make_model_final["class_id"] = make_model_final["make_model"].map(make_model_to_id)

print_split_summary(make_model_final, "make_model", "FINAL MAKE/MODEL SPLIT")

make_model_counts = (
    make_model_final.groupby(["class_id", "make", "model_name", "make_model", "split"])
    .size()
    .rename("images")
    .reset_index()
)

save_csv(make_model_final, "final_make_model_split.csv")
save_csv(make_model_counts, "final_make_model_class_counts.csv")

with open(OUTPUT_DIR / "final_make_model_class_to_id.json", "w", encoding="utf-8") as f:
    json.dump(make_model_to_id, f, indent=2, ensure_ascii=False)

print("Saved class mapping:", OUTPUT_DIR / "final_make_model_class_to_id.json")


Columns: ['file_path', 'make', 'model_name', 'body_type', 'width', 'height', 'source_dataset', 'image_hash', 'model_id', 'parent_brand']
Rows loaded: 135419
Exact duplicate rows removed: 0
Retained make/model classes: 1102
Retained images: 122099
Minimum retained class size: 50
Maximum retained class size: 303

FINAL MAKE/MODEL SPLIT
Rows: 122099
Classes: 1102

Split counts:
split
train    97641
val      12229
test     12229
Name: count, dtype: int64

Class count summary:
count    1102.000000
mean      110.797641
std        28.714636
min        50.000000
25%       103.000000
50%       119.000000
75%       120.000000
max       303.000000
dtype: float64
Saved: C:\Users\elesh\Academics\UTS\36127InnovationLab\Development\AutoVision_Model_Development\reports\final_dataset_preparation\final_make_model_split.csv
Saved: C:\Users\elesh\Academics\UTS\36127InnovationLab\Development\AutoVision_Model_Development\reports\final_dataset_preparation\final_make_model_class_counts.csv
Saved class mapping


## 4. Final body-type dataset


In [10]:
if WEB_METADATA is None:
    raise FileNotFoundError(
        "Could not find compcars_modeling_ready.csv."
    )

if "web_df" not in globals():
    web_df = pd.read_csv(WEB_METADATA)

required = {"file_path", "body_type"}
missing = required - set(web_df.columns)
if missing:
    raise KeyError(f"Missing required columns in web metadata: {sorted(missing)}")

body_df = web_df.dropna(subset=["file_path", "body_type"]).copy()

body_df["body_type"] = body_df["body_type"].astype(str).str.strip()
body_df = body_df[
    ~body_df["body_type"].isin(["", "nan", "None", "0"])
].copy()

if "image_hash" in body_df.columns:
    before = len(body_df)
    body_df = body_df.drop_duplicates(subset=["image_hash"]).copy()
    print("Exact duplicate rows removed:", before - len(body_df))

body_df = add_portable_path_column(
    body_df,
    source_col="file_path",
    dataset_key="compcars_web",
    markers=[
        "data/compcars_kaggle/extracted",
        "compcars_kaggle/extracted",
        "data/compcars_web",
        "compcars_web",
    ],
)

body_counts_raw = body_df["body_type"].value_counts()
too_small = body_counts_raw[body_counts_raw < 3]

if len(too_small):
    print("Dropping body-type labels with fewer than 3 samples:")
    print(too_small)
    body_df = body_df[~body_df["body_type"].isin(too_small.index)].copy()

body_final = deterministic_group_split(
    body_df,
    group_col="body_type",
    train_ratio=0.80,
    val_ratio=0.10,
    test_ratio=0.10,
    seed=SEED,
)

body_classes = sorted(body_final["body_type"].unique())
body_to_id = {name: i for i, name in enumerate(body_classes)}
body_final["class_id"] = body_final["body_type"].map(body_to_id)

print_split_summary(body_final, "body_type", "FINAL BODY-TYPE SPLIT")

body_counts = (
    body_final.groupby(["class_id", "body_type", "split"])
    .size()
    .rename("images")
    .reset_index()
)

save_csv(body_final, "final_body_type_split.csv")
save_csv(body_counts, "final_body_type_class_counts.csv")

with open(OUTPUT_DIR / "final_body_type_class_to_id.json", "w", encoding="utf-8") as f:
    json.dump(body_to_id, f, indent=2, ensure_ascii=False)

print("Saved class mapping:", OUTPUT_DIR / "final_body_type_class_to_id.json")


Exact duplicate rows removed: 0

FINAL BODY-TYPE SPLIT
Rows: 97518
Classes: 12

Split counts:
split
train    78016
val       9751
test      9751
Name: count, dtype: int64

Class count summary:
count       12.000000
mean      8126.500000
std      10027.994339
min        934.000000
25%       1904.250000
50%       3555.500000
75%       8745.250000
max      32232.000000
dtype: float64
Saved: C:\Users\elesh\Academics\UTS\36127InnovationLab\Development\AutoVision_Model_Development\reports\final_dataset_preparation\final_body_type_split.csv
Saved: C:\Users\elesh\Academics\UTS\36127InnovationLab\Development\AutoVision_Model_Development\reports\final_dataset_preparation\final_body_type_class_counts.csv
Saved class mapping: C:\Users\elesh\Academics\UTS\36127InnovationLab\Development\AutoVision_Model_Development\reports\final_dataset_preparation\final_body_type_class_to_id.json



## 5. Final colour dataset


In [14]:
if COLOUR_METADATA is None:
    raise FileNotFoundError(
        "Could not find compcars_surveillance_inventory_v2.csv. "
        "Set COLOUR_METADATA to its actual path in Section 2."
    )

colour_df = pd.read_csv(COLOUR_METADATA)

print("Columns:", colour_df.columns.tolist())
print("Rows loaded:", len(colour_df))

path_candidates = [
    "file_path",
    "image_path",
    "path",
    "filepath",
]

COLOUR_PATH_COL = next(
    (c for c in path_candidates if c in colour_df.columns),
    None,
)

if COLOUR_PATH_COL is None:
    raise KeyError(
        f"Could not identify image-path column. "
        f"Tried: {path_candidates}"
    )

colour_label_candidates = [
    "colour",
    "color",
    "colour_name",
    "color_name",
    "colour_id",
    "color_id",
]

COLOUR_LABEL_COL = next(
    (c for c in colour_label_candidates if c in colour_df.columns),
    None,
)

if COLOUR_LABEL_COL is None:
    raise KeyError(
        f"Could not identify colour label column. "
        f"Tried: {colour_label_candidates}"
    )


print("Using path column:", COLOUR_PATH_COL)
print("Using colour column:", COLOUR_LABEL_COL)


colour_final_source = colour_df.dropna(
    subset=[COLOUR_PATH_COL, COLOUR_LABEL_COL]
).copy()

COLOR_ID_TO_NAME = {
    0: "black",
    1: "white",
    2: "red",
    3: "yellow",
    4: "blue",
    5: "green",
    6: "purple",
    7: "brown",
    8: "champagne",
    9: "silver",
}


raw_colour = colour_final_source[COLOUR_LABEL_COL]


if pd.api.types.is_numeric_dtype(raw_colour):

    colour_final_source = colour_final_source[
        colour_final_source[COLOUR_LABEL_COL] != -1
    ].copy()

    colour_final_source["colour_original_id"] = (
        colour_final_source[COLOUR_LABEL_COL].astype(int)
    )

    colour_final_source["colour_label"] = (
        colour_final_source["colour_original_id"]
        .map(COLOR_ID_TO_NAME)
    )

else:

    invalid_names = {
        "-1",
        "unrecognized",
        "unrecognised",
        "unknown",
        "none",
        "nan",
        "",
    }

    colour_final_source = colour_final_source[
        ~colour_final_source[COLOUR_LABEL_COL]
        .astype(str)
        .str.strip()
        .str.lower()
        .isin(invalid_names)
    ].copy()

    colour_final_source["colour_label"] = (
        colour_final_source[COLOUR_LABEL_COL]
        .astype(str)
        .str.strip()
        .str.lower()
    )


if colour_final_source["colour_label"].isna().any():
    raise ValueError(
        "Some recognised colour IDs could not be mapped to colour names."
    )


print(
    "Recognised colour images:",
    len(colour_final_source)
)

print(
    "Colour classes:",
    sorted(colour_final_source["colour_label"].unique())
)


if "image_hash" in colour_final_source.columns:

    before = len(colour_final_source)

    colour_final_source = (
        colour_final_source
        .drop_duplicates(subset=["image_hash"])
        .copy()
    )

    print(
        "Exact duplicate rows removed:",
        before - len(colour_final_source)
    )

colour_final_source = add_portable_path_column(
    colour_final_source,
    source_col=COLOUR_PATH_COL,
    dataset_key="compcars_sv",
    markers=[
        "data/compcars_sv/extracted/sv_data",
        "compcars_sv/extracted/sv_data",
        "extracted/sv_data",
        "sv_data",
    ],
)

train_txt = (
    DEFAULT_SHARED_ROOT
    / "data"
    / "compcars_sv"
    / "extracted"
    / "sv_data"
    / "train_surveillance.txt"
)

test_txt = (
    DEFAULT_SHARED_ROOT
    / "data"
    / "compcars_sv"
    / "extracted"
    / "sv_data"
    / "test_surveillance.txt"
)


if not train_txt.exists():
    raise FileNotFoundError(
        f"Official training split not found:\n{train_txt}"
    )

if not test_txt.exists():
    raise FileNotFoundError(
        f"Official test split not found:\n{test_txt}"
    )


with open(train_txt, "r", encoding="utf-8") as f:
    official_train_paths = {
        line.strip().replace("\\", "/")
        for line in f
        if line.strip()
    }


with open(test_txt, "r", encoding="utf-8") as f:
    official_test_paths = {
        line.strip().replace("\\", "/")
        for line in f
        if line.strip()
    }


print(
    "Official train entries:",
    len(official_train_paths)
)

print(
    "Official test entries:",
    len(official_test_paths)
)


official_overlap = (
    official_train_paths
    & official_test_paths
)

print(
    "Official train/test overlap:",
    len(official_overlap)
)

if official_overlap:
    raise ValueError(
        "Official train and test lists overlap."
    )


colour_final_source["official_path_key"] = (
    colour_final_source["relative_path"]
    .astype(str)
    .str.replace("\\", "/", regex=False)
    .str.removeprefix("image/")
)


colour_final_source["official_split"] = pd.NA


colour_final_source.loc[
    colour_final_source["official_path_key"]
    .isin(official_train_paths),
    "official_split",
] = "train"


colour_final_source.loc[
    colour_final_source["official_path_key"]
    .isin(official_test_paths),
    "official_split",
] = "test"


print("\nMatched official split:")

print(
    colour_final_source["official_split"]
    .value_counts(dropna=False)
)

unmatched = colour_final_source[
    colour_final_source["official_split"].isna()
].copy()


print(
    "\nUnmatched recognised images:",
    len(unmatched)
)


if len(unmatched) > 0:

    print(
        unmatched[
            [
                "relative_path",
                "official_path_key",
            ]
        ].head(20)
    )

    raise ValueError(
        "Some recognised surveillance images could not "
        "be matched to the official CompCars split."
    )


official_train = colour_final_source[
    colour_final_source["official_split"] == "train"
].copy()


official_test = colour_final_source[
    colour_final_source["official_split"] == "test"
].copy()


print(
    "\nRecognised official training images:",
    len(official_train)
)

print(
    "Recognised official test images:",
    len(official_test)
)

train_val = deterministic_train_val_split(
    official_train,
    group_col="colour_label",
    train_ratio=0.90,
    val_ratio=0.10,
    seed=SEED,
)

official_test["split"] = "test"

colour_final = pd.concat(
    [
        train_val,
        official_test,
    ],
    ignore_index=True,
)


colour_split_method = (
    "Official CompCars surveillance train/test split; "
    "10% of official training reserved for validation"
)


print(
    "\nSplit method:",
    colour_split_method
)


colour_classes = sorted(
    colour_final["colour_label"].unique()
)


colour_to_id = {
    name: i
    for i, name in enumerate(colour_classes)
}


colour_final["class_id"] = (
    colour_final["colour_label"]
    .map(colour_to_id)
)


print("\nFinal colour mapping:")

for name, class_id in colour_to_id.items():
    print(
        f"{class_id}: {name}"
    )


print_split_summary(
    colour_final,
    "colour_label",
    "FINAL COLOUR SPLIT",
)

colour_counts = (
    colour_final
    .groupby(
        [
            "class_id",
            "colour_label",
            "split",
        ]
    )
    .size()
    .rename("images")
    .reset_index()
)


print("\nColour distribution by split:")

display(
    colour_counts
    .pivot(
        index=[
            "class_id",
            "colour_label",
        ],
        columns="split",
        values="images",
    )
    .fillna(0)
)


absolute_windows_paths = (
    colour_final["relative_path"]
    .str.contains(
        r"^[A-Za-z]:",
        regex=True,
    )
    .sum()
)


print(
    "\nAbsolute Windows paths:",
    absolute_windows_paths
)


if absolute_windows_paths != 0:
    raise ValueError(
        "Colour split still contains Windows absolute paths."
    )

colour_final = (
    colour_final
    .drop(
        columns=["official_path_key"],
        errors="ignore",
    )
)

save_csv(
    colour_final,
    "final_colour_split.csv",
)


save_csv(
    colour_counts,
    "final_colour_class_counts.csv",
)


with open(
    OUTPUT_DIR
    / "final_colour_class_to_id.json",
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        colour_to_id,
        f,
        indent=2,
        ensure_ascii=False,
    )


print(
    "Saved class mapping:",
    OUTPUT_DIR
    / "final_colour_class_to_id.json",
)

Columns: ['file_path', 'surveillance_model_id', 'make', 'model', 'web_model_id', 'color', 'width', 'height', 'corrupted', 'image_hash', 'source_dataset', 'body_type', 'parent_brand']
Rows loaded: 44481
Using path column: file_path
Using colour column: color
Recognised colour images: 44441
Colour classes: ['black', 'blue', 'brown', 'champagne', 'green', 'purple', 'red', 'silver', 'white', 'yellow']
Exact duplicate rows removed: 0
Official train entries: 31148
Official test entries: 13333
Official train/test overlap: 0

Matched official split:
official_split
train    31118
test     13323
Name: count, dtype: int64

Unmatched recognised images: 0

Recognised official training images: 31118
Recognised official test images: 13323

Split method: Official CompCars surveillance train/test split; 10% of official training reserved for validation

Final colour mapping:
0: black
1: blue
2: brown
3: champagne
4: green
5: purple
6: red
7: silver
8: white
9: yellow

FINAL COLOUR SPLIT
Rows: 44441
Clas

,split,test,train,val
class_id,colour_label,,,
0,black,4475,9392,1044
1,blue,569,1176,131
2,brown,441,917,102
3,champagne,244,527,58
4,green,86,152,17
5,purple,64,141,16
6,red,1085,2288,254
7,silver,3750,7937,882
8,white,2259,4780,531



Absolute Windows paths: 0
Saved: C:\Users\elesh\Academics\UTS\36127InnovationLab\Development\AutoVision_Model_Development\reports\final_dataset_preparation\final_colour_split.csv
Saved: C:\Users\elesh\Academics\UTS\36127InnovationLab\Development\AutoVision_Model_Development\reports\final_dataset_preparation\final_colour_class_counts.csv
Saved class mapping: C:\Users\elesh\Academics\UTS\36127InnovationLab\Development\AutoVision_Model_Development\reports\final_dataset_preparation\final_colour_class_to_id.json



## 6. Final damage-detection dataset


In [24]:
NEGATIVE_AWARE_ROOT = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "cardd_with_undamaged_negatives"
)

CARDD_ROOT = (
    DEFAULT_SHARED_ROOT
    / "data"
    / "cardd_kaggle"
    / "extracted"
)

CARDD_INVENTORY = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "cardd_inventory.csv"
)

CLEAN_HOLDOUT_MANIFEST = (
    PROJECT_ROOT
    / "reports"
    / "damage_negative_training"
    / "clean_holdout_manifest.csv"
)

required_paths = [
    NEGATIVE_AWARE_ROOT / "train" / "images",
    NEGATIVE_AWARE_ROOT / "train" / "labels",
    NEGATIVE_AWARE_ROOT / "val" / "images",
    NEGATIVE_AWARE_ROOT / "val" / "labels",
    CARDD_ROOT / "test" / "images",
    CARDD_ROOT / "test" / "labels",
    CARDD_INVENTORY,
    CLEAN_HOLDOUT_MANIFEST,
]

for p in required_paths:
    if not p.exists():
        raise FileNotFoundError(
            f"Required damage dataset path not found:\n{p}"
        )

print("All required damage paths found.")

IMAGE_EXTENSIONS = {
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".webp",
}


def list_images(folder):
    folder = Path(folder)

    return sorted(
        p
        for p in folder.rglob("*")
        if p.is_file()
        and p.suffix.lower() in IMAGE_EXTENSIONS
    )


def build_processed_split_manifest(
    dataset_root,
    split,
):
    image_root = dataset_root / split / "images"
    label_root = dataset_root / split / "labels"

    rows = []

    for image_path in list_images(image_root):
        relative_image = image_path.relative_to(
            dataset_root
        )

        relative_inside_images = image_path.relative_to(
            image_root
        )

        label_path = (
            label_root
            / relative_inside_images.with_suffix(".txt")
        )

        if not label_path.exists():
            raise FileNotFoundError(
                f"Missing YOLO label for:\n{image_path}"
            )

        has_damage = label_path.stat().st_size > 0

        source = (
            "CarDD"
            if has_damage
            else "explicit_clean_negative"
        )

        rows.append(
            {
                "dataset_key": "damage_negative_aware",
                "source": source,
                "split": split,
                "relative_path": str(
                    relative_image
                ).replace("\\", "/"),
                "label_relative_path": str(
                    label_path.relative_to(
                        dataset_root
                    )
                ).replace("\\", "/"),
                "has_damage": has_damage,
                "is_clean_holdout": False,
            }
        )

    return pd.DataFrame(rows)


damage_train = build_processed_split_manifest(
    NEGATIVE_AWARE_ROOT,
    "train",
)

damage_val = build_processed_split_manifest(
    NEGATIVE_AWARE_ROOT,
    "val",
)

print(
    "\nNegative-aware training images:",
    len(damage_train),
)

print(
    damage_train["source"].value_counts()
)

print(
    "\nNegative-aware validation images:",
    len(damage_val),
)

print(
    damage_val["source"].value_counts()
)


cardd_inventory = pd.read_csv(
    CARDD_INVENTORY
)

original_test = cardd_inventory[
    cardd_inventory["split"] == "test"
].copy()

print(
    "\nOriginal CarDD test images:",
    len(original_test),
)

if len(original_test) != 374:
    raise ValueError(
        f"Expected 374 original CarDD test images, "
        f"found {len(original_test)}."
    )


test_rows = []

for _, row in original_test.iterrows():
    original_path = str(
        row["file_path"]
    ).replace("\\", "/")

    marker = "cardd_kaggle/extracted/"

    if marker not in original_path:
        raise ValueError(
            f"Could not convert original CarDD path:\n"
            f"{original_path}"
        )

    relative_image = original_path.split(
        marker,
        1,
    )[1]

    relative_label = relative_image.replace(
        "/images/",
        "/labels/",
    )

    relative_label = str(
        Path(relative_label).with_suffix(".txt")
    ).replace("\\", "/")

    real_image_path = (
        CARDD_ROOT
        / Path(relative_image)
    )

    real_label_path = (
        CARDD_ROOT
        / Path(relative_label)
    )

    if not real_image_path.exists():
        raise FileNotFoundError(
            f"CarDD test image not found:\n"
            f"{real_image_path}"
        )

    if not real_label_path.exists():
        raise FileNotFoundError(
            f"CarDD test label not found:\n"
            f"{real_label_path}"
        )

    test_rows.append(
        {
            "dataset_key": "cardd_original",
            "source": "CarDD",
            "split": "test",
            "relative_path": relative_image,
            "label_relative_path": relative_label,
            "has_damage": True,
            "is_clean_holdout": False,
        }
    )


damage_test = pd.DataFrame(
    test_rows
)

print(
    "Verified original CarDD test:",
    len(damage_test),
)


raw_holdout = pd.read_csv(
    CLEAN_HOLDOUT_MANIFEST
)

print(
    "\nClean holdout manifest columns:",
    raw_holdout.columns.tolist(),
)

print(
    "Clean holdout rows:",
    len(raw_holdout),
)

if "holdout_path" not in raw_holdout.columns:
    raise KeyError(
        "Expected 'holdout_path' column inside "
        "clean_holdout_manifest.csv"
    )


clean_rows = []

for _, row in raw_holdout.iterrows():
    holdout_path = Path(
        str(row["holdout_path"])
    )

    try:
        relative_path = str(
            holdout_path.resolve().relative_to(
                PROJECT_ROOT.resolve()
            )
        ).replace("\\", "/")

        dataset_key = "project"

    except ValueError:
        relative_path = (
            "clean_holdout/"
            + holdout_path.name
        )

        dataset_key = "clean_holdout"

    clean_rows.append(
        {
            "dataset_key": dataset_key,
            "source": "explicit_clean_negative",
            "split": "clean_holdout",
            "relative_path": relative_path,
            "label_relative_path": None,
            "has_damage": False,
            "is_clean_holdout": True,
        }
    )


clean_holdout = pd.DataFrame(
    clean_rows
)

print(
    "Prepared clean holdout:",
    len(clean_holdout),
)

if len(clean_holdout) != 120:
    raise ValueError(
        f"Expected 120 clean holdout images, "
        f"found {len(clean_holdout)}."
    )


final_damage_manifest = pd.concat(
    [
        damage_train,
        damage_val,
        damage_test,
        clean_holdout,
    ],
    ignore_index=True,
)


train_cardd = (
    damage_train["source"]
    .eq("CarDD")
    .sum()
)

train_clean = (
    damage_train["source"]
    .eq("explicit_clean_negative")
    .sum()
)

val_cardd = (
    damage_val["source"]
    .eq("CarDD")
    .sum()
)

val_clean = (
    damage_val["source"]
    .eq("explicit_clean_negative")
    .sum()
)


print(
    "\n"
    + "=" * 80
)

print(
    "FINAL DAMAGE DATASET"
)

print(
    "=" * 80
)

print(
    "\nSplit counts:"
)

print(
    final_damage_manifest[
        "split"
    ].value_counts()
)

print(
    "\nSource by split:"
)

print(
    final_damage_manifest.groupby(
        [
            "split",
            "source",
        ]
    ).size()
)

print(
    "\nExpected composition:"
)

print(
    f"Train CarDD: {train_cardd}"
)

print(
    f"Train clean: {train_clean}"
)

print(
    f"Val CarDD: {val_cardd}"
)

print(
    f"Val clean: {val_clean}"
)

print(
    f"CarDD test: {len(damage_test)}"
)

print(
    f"Clean holdout: {len(clean_holdout)}"
)


assert train_cardd == 2816
assert train_clean == 600
assert val_cardd == 810
assert val_clean == 80
assert len(damage_test) == 374
assert len(clean_holdout) == 120


save_csv(
    final_damage_manifest,
    "final_damage_manifest.csv",
)

damage_summary = (
    final_damage_manifest
    .groupby(
        [
            "split",
            "source",
            "has_damage",
        ],
        dropna=False,
    )
    .size()
    .rename("images")
    .reset_index()
)

save_csv(
    damage_summary,
    "final_damage_summary.csv",
)

display(
    damage_summary
)

print(
    "\nDamage dataset preparation complete."
)

All required damage paths found.

Negative-aware training images: 3416
source
CarDD                      2816
explicit_clean_negative     600
Name: count, dtype: int64

Negative-aware validation images: 890
source
CarDD                      810
explicit_clean_negative     80
Name: count, dtype: int64

Original CarDD test images: 374
Verified original CarDD test: 374

Clean holdout manifest columns: ['source_path', 'holdout_path']
Clean holdout rows: 120
Prepared clean holdout: 120

FINAL DAMAGE DATASET

Split counts:
split
train            3416
val               890
test              374
clean_holdout     120
Name: count, dtype: int64

Source by split:
split          source                 
clean_holdout  explicit_clean_negative     120
test           CarDD                       374
train          CarDD                      2816
               explicit_clean_negative     600
val            CarDD                       810
               explicit_clean_negative      80
dtype: int64

Expe

,split,source,has_damage,images
0,clean_holdout,explicit_clean_negative,False,120
1,test,CarDD,True,374
2,train,CarDD,True,2816
3,train,explicit_clean_negative,False,600
4,val,CarDD,True,810
5,val,explicit_clean_negative,False,80



Damage dataset preparation complete.



## 7. Final integrity checks

In [25]:
def check_path_split_leakage(df, name):
    if df is None or df.empty:
        print(f"{name}: skipped (empty)")
        return

    duplicated = (
        df.groupby(["dataset_key", "relative_path"])["split"]
        .nunique()
    )

    leaked = duplicated[duplicated > 1]

    print(f"{name}: paths appearing in multiple splits = {len(leaked)}")

    if len(leaked):
        display(leaked.head(20))
        raise AssertionError(f"{name} contains split leakage.")

def check_all_classes_in_train(df, label_col, name):
    if df is None or df.empty:
        return

    all_classes = set(df[label_col].unique())
    train_classes = set(df.loc[df["split"] == "train", label_col].unique())
    missing = sorted(all_classes - train_classes)

    print(f"{name}: classes missing from train = {len(missing)}")

    if missing:
        print(missing[:20])
        raise AssertionError(f"{name}: some classes are absent from training.")

if "make_model_final" in globals():
    check_path_split_leakage(make_model_final, "make/model")
    check_all_classes_in_train(make_model_final, "make_model", "make/model")

if "body_final" in globals():
    check_path_split_leakage(body_final, "body type")
    check_all_classes_in_train(body_final, "body_type", "body type")

if "colour_final" in globals():
    check_path_split_leakage(colour_final, "colour")
    check_all_classes_in_train(colour_final, "colour_label", "colour")

if "final_damage_manifest" in globals() and not final_damage_manifest.empty:
    check_path_split_leakage(final_damage_manifest, "damage")

    clean_holdout = final_damage_manifest[
        final_damage_manifest["split"] == "clean_holdout"
    ]

    assert not clean_holdout["has_damage"].any(), (
        "Clean holdout unexpectedly contains damage-labelled rows."
    )

    print(
        "damage: clean holdout is separate and contains",
        len(clean_holdout),
        "images"
    )

print("\nIntegrity checks completed.")


make/model: paths appearing in multiple splits = 0
make/model: classes missing from train = 0
body type: paths appearing in multiple splits = 0
body type: classes missing from train = 0
colour: paths appearing in multiple splits = 0
colour: classes missing from train = 0
damage: paths appearing in multiple splits = 0
damage: clean holdout is separate and contains 120 images

Integrity checks completed.


## 8. Saving overall dataset summary

In [26]:
summary_rows = []

def add_summary(task, df, label_col=None, note=""):
    if df is None or df.empty:
        return

    for split, count in df["split"].value_counts().items():
        row = {
            "task": task,
            "split": split,
            "images": int(count),
            "classes": (
                int(df[label_col].nunique())
                if label_col is not None else np.nan
            ),
            "note": note,
        }
        summary_rows.append(row)

if "make_model_final" in globals():
    add_summary(
        "make_model",
        make_model_final,
        "make_model",
        f"ResNet50; min {MIN_IMAGES_PER_MAKE_MODEL} images per retained class",
    )

if "body_final" in globals():
    add_summary(
        "body_type",
        body_final,
        "body_type",
        "ConvNeXt-Tiny; all valid labelled CompCars web rows",
    )

if "colour_final" in globals():
    add_summary(
        "colour",
        colour_final,
        "colour_label",
        f"Swin-T; {colour_split_method}",
    )

if "final_damage_manifest" in globals() and not final_damage_manifest.empty:
    add_summary(
        "damage",
        final_damage_manifest,
        None,
        "YOLO11s; original CarDD split + explicit clean negatives",
    )

final_summary = pd.DataFrame(summary_rows)

if not final_summary.empty:
    display(final_summary)
    save_csv(final_summary, "final_dataset_summary.csv")
else:
    print("No completed datasets were available for the overall summary.")


,task,split,images,classes,note
0,make_model,train,97641,1102.0,ResNet50; min 50 images per retained class
1,make_model,val,12229,1102.0,ResNet50; min 50 images per retained class
2,make_model,test,12229,1102.0,ResNet50; min 50 images per retained class
3,body_type,train,78016,12.0,ConvNeXt-Tiny; all valid labelled CompCars web...
4,body_type,val,9751,12.0,ConvNeXt-Tiny; all valid labelled CompCars web...
5,body_type,test,9751,12.0,ConvNeXt-Tiny; all valid labelled CompCars web...
6,colour,train,28006,10.0,Swin-T; Official CompCars surveillance train/t...
7,colour,test,13323,10.0,Swin-T; Official CompCars surveillance train/t...
8,colour,val,3112,10.0,Swin-T; Official CompCars surveillance train/t...
9,damage,train,3416,NaN,YOLO11s; original CarDD split + explicit clean...


Saved: C:\Users\elesh\Academics\UTS\36127InnovationLab\Development\AutoVision_Model_Development\reports\final_dataset_preparation\final_dataset_summary.csv
